In [ ]:
# ============================================================
# GOOGLE COLAB CODE
# Ambush Hypothesis Analysis for 15-mers
#
# This code:
# 1. Reads CDS sequences from CSV
# 2. Uses Biopython translation tables
# 3. Checks stop codons in:
#       original frame
#       +1 shifted frame
#       +2 shifted frame
# 4. Counts mapped 15mers for +1 and +2 shifts
# 5. Finds stop codons before the first mapped 15mer
# 6. Finds stop codons inside mapped 15mers
# 7. Performs paired t-test + Bonferroni correction
# 8. Saves detailed results, summary files, and inferential statistics
# ============================================================

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

!pip install biopython statsmodels -q

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from Bio.Data import CodonTable
import re

from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
from itertools import combinations


# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

import os
import shutil

if os.path.exists("/content/drive") and not os.path.exists("/content/drive/MyDrive"):
    shutil.rmtree("/content/drive")

from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")


# ============================================================
# FILE PATHS
# ============================================================

input_file = "/content/drive/MyDrive/final_pep_frames_tranls.csv"

output_file = "/content/ambush_hypothesis_results_with_first15mer.csv"
summary_output_file = "/content/ambush_hypothesis_summary_with_first15mer.csv"

mapped15mer_output_file = "/content/mapped15mer_stopcodon_analysis.csv"
mapped15mer_summary_file = "/content/mapped15mer_stopcodon_summary.csv"

inferential_output_file = "/content/stop_count_ttest_bonferroni_results_15mer.csv"

# ============================================================
# CLEAN FUNCTIONS
# ============================================================

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = seq.replace("U", "T")
    seq = re.sub(r"[^ATGC]", "", seq)
    return seq


def normalize_shift(x):
    x = str(x).strip()

    if x in ["1", "1.0"]:
        return "1"

    if x in ["2", "2.0"]:
        return "2"

    if x in ["transl_table_issue", "trans_table_issue"]:
        return "transl_table_issue"

    return x


def normalize_transl_table(x):
    try:
        return int(float(str(x).strip()))
    except Exception:
        return None


def clean_position_plus_one(pos):
    try:
        pos = str(pos).strip()

        if "|" in pos:
            pos = pos.split("|")[0]

        return int(float(pos)) + 1

    except Exception:
        return None


def get_shifted_codons(seq, shift):
    shifted_seq = seq[shift:]
    codons = []

    for i in range(0, len(shifted_seq) - 2, 3):
        codon = shifted_seq[i:i + 3]

        if len(codon) == 3:
            codons.append(codon)

    return codons


def find_stop_codons(codons, stop_codons):
    stop_positions = []
    stop_codon_list = []

    for idx, codon in enumerate(codons):
        if codon in stop_codons:
            stop_positions.append(idx + 1)
            stop_codon_list.append(codon)

    return stop_codon_list, stop_positions


def count_stop_codons_before_nt_position(seq, frame_shift, stop_codons, nt_position):
    """
    Counts stop codons before the first mapped 15mer.

    nt_position is 1-based nucleotide position.
    frame_shift:
        0 = original frame
        1 = +1 frame
        2 = +2 frame
    """

    if nt_position is None or pd.isna(nt_position):
        return 0, "", ""

    codons = get_shifted_codons(seq, frame_shift)

    stop_count = 0
    stop_codon_list = []
    stop_position_list = []

    for idx, codon in enumerate(codons):
        codon_start_0based = frame_shift + idx * 3
        codon_start_1based = codon_start_0based + 1

        if codon_start_1based >= nt_position:
            break

        if codon in stop_codons:
            stop_count += 1
            stop_codon_list.append(codon)
            stop_position_list.append(idx + 1)

    return (
        stop_count,
        "|".join(stop_codon_list),
        "|".join(map(str, stop_position_list))
    )


def count_stop_codons_inside_15mer_all_frames(seq15, stop_codons):
    """
    Counts stop codons inside a 15mer across internal frames 0, 1, and 2.
    """

    seq15 = clean_sequence(seq15)

    stop_count = 0
    found_stop_codons = []
    found_internal_frames = []
    found_positions = []

    for frame in [0, 1, 2]:
        codons = get_shifted_codons(seq15, frame)

        for idx, codon in enumerate(codons):
            if codon in stop_codons:
                stop_count += 1
                found_stop_codons.append(codon)
                found_internal_frames.append(str(frame))
                found_positions.append(str(idx + 1))

    return (
        stop_count,
        "|".join(found_stop_codons),
        "|".join(found_internal_frames),
        "|".join(found_positions)
    )


# ============================================================
# LOAD DATA
# ============================================================

print("\nLoading dataset...")

df = pd.read_csv(
    input_file,
    dtype=str,
    low_memory=False
)

print("\nDataset loaded.")
print("Total rows:", len(df))

# ============================================================
# CLEAN DATA
# ============================================================

print("\nCleaning dataset...")

df["shift"] = df["shift"].apply(normalize_shift)
df["cds_sequence_clean"] = df["cds_sequence"].apply(clean_sequence)
df["matched_15mer_clean"] = df["matched_15mer"].apply(clean_sequence)
df["transl_table_clean"] = df["transl_table"].apply(normalize_transl_table)
df["position_corrected_plus_one"] = df["positions"].apply(clean_position_plus_one)

print("\nShift distribution:")
print(df["shift"].value_counts(dropna=False))

print("\nTranslation table distribution:")
print(df["transl_table_clean"].value_counts(dropna=False))

# ============================================================
# BUILD UNIQUE CDS TABLE
# ============================================================

print("\nBuilding unique CDS table...")

unique_cds = df[
    ["cds_sequence_clean", "transl_table_clean"]
].drop_duplicates()

unique_cds = unique_cds.dropna(
    subset=["transl_table_clean"]
)

unique_cds["transl_table_clean"] = unique_cds[
    "transl_table_clean"
].astype(int)

print(
    "Unique CDS + translation table combinations:",
    len(unique_cds)
)

# ============================================================
# COUNT MAPPED 15MERS AND FIRST MAPPED 15MER POSITION
# ============================================================

print("\nCounting mapped 15mers and first mapped 15mer position...")

shift1_counts = defaultdict(int)
shift2_counts = defaultdict(int)
transl_table_issue_counts = defaultdict(int)
all_mapped_counts = defaultdict(int)

first_mapped_position = {}

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Counting mapped 15mers"
):

    cds = row["cds_sequence_clean"]
    shift = row["shift"]
    pos = row["position_corrected_plus_one"]

    if shift == "1":
        shift1_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "2":
        shift2_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "transl_table_issue":
        transl_table_issue_counts[cds] += 1
        all_mapped_counts[cds] += 1

    if pos is not None and not pd.isna(pos):
        if cds not in first_mapped_position:
            first_mapped_position[cds] = pos
        else:
            first_mapped_position[cds] = min(
                first_mapped_position[cds],
                pos
            )

# ============================================================
# MAIN CDS-LEVEL AMBUSH ANALYSIS
# ============================================================

print(
    "\nRunning original-frame, +1-frame, +2-frame, "
    "and before-first-15mer stop codon analysis..."
)

results = []
invalid_translation_tables = []

for _, row in tqdm(
    unique_cds.iterrows(),
    total=len(unique_cds),
    desc="CDS-level stop codon analysis"
):

    cds_seq = row["cds_sequence_clean"]
    transl_table_id = row["transl_table_clean"]

    try:
        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        invalid_translation_tables.append(transl_table_id)
        continue

    # Original frame
    original_codons = get_shifted_codons(cds_seq, 0)
    original_stop_codons, original_stop_positions = find_stop_codons(
        original_codons,
        stop_codons
    )

    # +1 frame
    plus1_codons = get_shifted_codons(cds_seq, 1)
    plus1_stop_codons, plus1_stop_positions = find_stop_codons(
        plus1_codons,
        stop_codons
    )

    # +2 frame
    plus2_codons = get_shifted_codons(cds_seq, 2)
    plus2_stop_codons, plus2_stop_positions = find_stop_codons(
        plus2_codons,
        stop_codons
    )

    # Stop codons before first mapped 15mer
    first_pos = first_mapped_position.get(cds_seq, np.nan)

    original_before_count, original_before_codons, original_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            0,
            stop_codons,
            first_pos
        )
    )

    plus1_before_count, plus1_before_codons, plus1_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            1,
            stop_codons,
            first_pos
        )
    )

    plus2_before_count, plus2_before_codons, plus2_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            2,
            stop_codons,
            first_pos
        )
    )

    results.append({

        "cds_sequence": cds_seq,
        "translation_table": transl_table_id,
        "stop_codons_for_table": "|".join(sorted(stop_codons)),
        "cds_length_nt": len(cds_seq),
        "first_mapped_15mer_nt_position": first_pos,

        # Original frame
        "original_total_codons": len(original_codons),
        "original_stop_count": len(original_stop_codons),
        "original_stop_density_per_100_codons": (
            len(original_stop_codons) / len(original_codons) * 100
        ) if len(original_codons) > 0 else np.nan,
        "original_stop_codons": "|".join(original_stop_codons),
        "original_stop_positions": "|".join(map(str, original_stop_positions)),

        # +1 frame
        "plus1_total_codons": len(plus1_codons),
        "plus1_stop_count": len(plus1_stop_codons),
        "plus1_stop_density_per_100_codons": (
            len(plus1_stop_codons) / len(plus1_codons) * 100
        ) if len(plus1_codons) > 0 else np.nan,
        "plus1_stop_codons": "|".join(plus1_stop_codons),
        "plus1_stop_positions": "|".join(map(str, plus1_stop_positions)),
        "plus1_mapped_15mers": shift1_counts.get(cds_seq, 0),

        # +2 frame
        "plus2_total_codons": len(plus2_codons),
        "plus2_stop_count": len(plus2_stop_codons),
        "plus2_stop_density_per_100_codons": (
            len(plus2_stop_codons) / len(plus2_codons) * 100
        ) if len(plus2_codons) > 0 else np.nan,
        "plus2_stop_codons": "|".join(plus2_stop_codons),
        "plus2_stop_positions": "|".join(map(str, plus2_stop_positions)),
        "plus2_mapped_15mers": shift2_counts.get(cds_seq, 0),

        # Mapped 15mer summary
        "transl_table_issue_mapped_15mers": transl_table_issue_counts.get(cds_seq, 0),

        "total_mapped_15mers_shift1_shift2": (
            shift1_counts.get(cds_seq, 0)
            +
            shift2_counts.get(cds_seq, 0)
        ),

        "total_mapped_15mers_all_tracked_groups": all_mapped_counts.get(cds_seq, 0),

        # Stop codons before first mapped 15mer
        "original_stop_count_before_first_mapped_15mer": original_before_count,
        "original_stop_codons_before_first_mapped_15mer": original_before_codons,
        "original_stop_positions_before_first_mapped_15mer": original_before_positions,

        "plus1_stop_count_before_first_mapped_15mer": plus1_before_count,
        "plus1_stop_codons_before_first_mapped_15mer": plus1_before_codons,
        "plus1_stop_positions_before_first_mapped_15mer": plus1_before_positions,

        "plus2_stop_count_before_first_mapped_15mer": plus2_before_count,
        "plus2_stop_codons_before_first_mapped_15mer": plus2_before_codons,
        "plus2_stop_positions_before_first_mapped_15mer": plus2_before_positions
    })

# ============================================================
# MAPPED 15MER-LEVEL STOP CODON ANALYSIS
# ============================================================

print("\nRunning mapped 15mer-level stop codon analysis...")

mapped15_results = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Mapped 15mer stop codon analysis"
):

    transl_table_id = row["transl_table_clean"]

    if pd.isna(transl_table_id):
        continue

    try:
        transl_table_id = int(transl_table_id)

        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        continue

    seq15 = row["matched_15mer_clean"]

    stop_count, stop_codons_found, internal_frames, internal_positions = (
        count_stop_codons_inside_15mer_all_frames(
            seq15,
            stop_codons
        )
    )

    mapped15_results.append({

        "cds_sequence_clean": row["cds_sequence_clean"],
        "matched_15mer_clean": seq15,
        "shift": row["shift"],
        "translation_table": transl_table_id,

        "mapped_15mer_has_stop_codon": stop_count > 0,
        "mapped_15mer_stop_codon_count": stop_count,
        "mapped_15mer_stop_codons": stop_codons_found,
        "mapped_15mer_stop_internal_frames": internal_frames,
        "mapped_15mer_stop_internal_positions": internal_positions
    })

# ============================================================
# SAVE DETAILED OUTPUTS
# ============================================================

results_df = pd.DataFrame(results)
mapped15_df = pd.DataFrame(mapped15_results)

# ============================================================
# INFERENTIAL STATISTICS
# Paired t-test + Bonferroni correction
# Comparing original_stop_count, plus1_stop_count, plus2_stop_count
# ============================================================

stop_count_cols = [
    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count"
]

stats_df = results_df[stop_count_cols].copy()

for col in stop_count_cols:
    stats_df[col] = pd.to_numeric(stats_df[col], errors="coerce")

stats_df = stats_df.dropna()

ttest_results = []

for col1, col2 in combinations(stop_count_cols, 2):

    t_stat, p_value = ttest_rel(
        stats_df[col1],
        stats_df[col2],
        nan_policy="omit"
    )

    ttest_results.append({
        "Comparison": f"{col1} vs {col2}",
        "Variable_1": col1,
        "Variable_2": col2,
        "Mean_1": stats_df[col1].mean(),
        "SD_1": stats_df[col1].std(ddof=1),
        "Mean_2": stats_df[col2].mean(),
        "SD_2": stats_df[col2].std(ddof=1),
        "N": len(stats_df),
        "t_statistic": t_stat,
        "raw_p_value": p_value
    })

inferential_df = pd.DataFrame(ttest_results)

reject, corrected_pvals, _, _ = multipletests(
    inferential_df["raw_p_value"],
    alpha=0.05,
    method="bonferroni"
)

inferential_df["bonferroni_corrected_p"] = corrected_pvals
inferential_df["significant_after_bonferroni_95CI"] = reject

inferential_df.to_csv(
    inferential_output_file,
    index=False
)

print("\nInferential statistics saved:")
print(inferential_output_file)

print("\nStop-count paired t-test + Bonferroni results:")
print(inferential_df)

# ============================================================
# SAVE MAIN OUTPUT FILES
# ============================================================

results_df.to_csv(
    output_file,
    index=False
)

mapped15_df.to_csv(
    mapped15mer_output_file,
    index=False
)

print("\nCDS-level detailed output saved:")
print(output_file)

print("\nMapped 15mer-level output saved:")
print(mapped15mer_output_file)

print("\nTotal CDS analyzed:")
print(len(results_df))

print("\nTotal mapped 15mers analyzed:")
print(len(mapped15_df))

# ============================================================
# CDS-LEVEL SUMMARY STATISTICS
# ============================================================

summary_cols = [

    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count",

    "original_stop_density_per_100_codons",
    "plus1_stop_density_per_100_codons",
    "plus2_stop_density_per_100_codons",

    "original_stop_count_before_first_mapped_15mer",
    "plus1_stop_count_before_first_mapped_15mer",
    "plus2_stop_count_before_first_mapped_15mer"
]

summary_df = results_df[
    summary_cols
].describe()

summary_df.to_csv(
    summary_output_file
)

print("\nCDS-level summary saved:")
print(summary_output_file)

# ============================================================
# MAPPED 15MER STOP SUMMARY
# ============================================================

total_15mers = len(mapped15_df)

stop_containing_15mers = mapped15_df[
    mapped15_df["mapped_15mer_has_stop_codon"] == True
]

percent_15mers_with_stop = (
    len(stop_containing_15mers)
    / total_15mers
    * 100
    if total_15mers > 0
    else np.nan
)

average_stop_codons_all_15mers = mapped15_df[
    "mapped_15mer_stop_codon_count"
].mean()

average_stop_codons_stop_containing_only = stop_containing_15mers[
    "mapped_15mer_stop_codon_count"
].mean()

mapped15_summary = pd.DataFrame({

    "metric": [

        "total_mapped_15mers",
        "mapped_15mers_with_at_least_one_stop",
        "percent_mapped_15mers_with_stop",
        "average_stop_codons_per_15mer_all",
        "average_stop_codons_per_stop_containing_15mer_only"

    ],

    "value": [

        total_15mers,
        len(stop_containing_15mers),
        percent_15mers_with_stop,
        average_stop_codons_all_15mers,
        average_stop_codons_stop_containing_only
    ]
})

mapped15_summary.to_csv(
    mapped15mer_summary_file,
    index=False
)

print("\nMapped 15mer stop summary saved:")
print(mapped15mer_summary_file)

# ============================================================
# INVALID TRANSLATION TABLES
# ============================================================

if len(invalid_translation_tables) > 0:
    print("\nInvalid / unsupported translation table IDs encountered:")
    print(sorted(set(invalid_translation_tables)))

# ============================================================
# PREVIEW
# ============================================================

print("\nPreview of CDS-level detailed results:")
print(results_df.head())

print("\nCDS-level summary statistics:")
print(summary_df)

print("\nMapped 15mer stop summary:")
print(mapped15_summary)

print("\nInferential statistics:")
print(inferential_df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.9 MB/s eta 0:00:00
Mounted at /content/drive

Loading dataset...

Dataset loaded.
Total rows: 1423523

Cleaning dataset...

Shift distribution:
shift
1                     761327
2                     659683
nan                     2498
transl_table_issue        15
Name: count, dtype: int64

Translation table distribution:
transl_table_clean
1     747497
11    633580
4      24467
12     10146
2       4958
6       1017
5       1003
3        656
10        95
9         91
28         8
15         2
16         2
25         1
Name: count, dtype: int64

Building unique CDS table...
Unique CDS + translation table combinations: 377524

Counting mapped 15mers and first mapped 15mer position...


Counting mapped 15mers: 100%|██████████| 1423523/1423523 [01:33<00:00, 15305.75it/s]



Running original-frame, +1-frame, +2-frame, and before-first-15mer stop codon analysis...


CDS-level stop codon analysis: 100%|██████████| 377524/377524 [06:51<00:00, 917.72it/s]



Running mapped 15mer-level stop codon analysis...


Mapped 15mer stop codon analysis: 100%|██████████| 1423523/1423523 [02:14<00:00, 10560.16it/s]



Inferential statistics saved:
/content/stop_count_ttest_bonferroni_results_15mer.csv

Stop-count paired t-test + Bonferroni results:
                                Comparison           Variable_1  \
0  original_stop_count vs plus1_stop_count  original_stop_count   
1  original_stop_count vs plus2_stop_count  original_stop_count   
2     plus1_stop_count vs plus2_stop_count     plus1_stop_count   

         Variable_2     Mean_1       SD_1     Mean_2       SD_2       N  \
0  plus1_stop_count   1.117842   3.207698  23.072811  23.504066  377524   
1  plus2_stop_count   1.117842   3.207698  28.999179  38.099454  377524   
2  plus2_stop_count  23.072811  23.504066  28.999179  38.099454  377524   

   t_statistic  raw_p_value  bonferroni_corrected_p  \
0  -577.519422          0.0                     0.0   
1  -449.696683          0.0                     0.0   
2  -167.274932          0.0                     0.0   

   significant_after_bonferroni_95CI  
0                               True

In [ ]:
# ============================================================
# GOOGLE COLAB CODE
# Ambush Hypothesis Analysis for 12-mers / matched peptide mers
#
# Fixed version:
# - Automatically detects matched_12mer / matched_15mer / similar column
# - Prevents KeyError: 'matched_12mer'
# - Counts stop codons in original, +1, +2 frames
# - Counts stop codons before first mapped mer
# - Counts stop codons inside mapped mer
# - Performs paired t-test + Bonferroni correction
# ============================================================

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

!pip install biopython statsmodels -q

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from Bio.Data import CodonTable
import re

from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
from itertools import combinations


# ============================================================
# FILE PATHS
# ============================================================

input_file = "/content/CQWW_frames.csv"

output_file = "/content/ambush_hypothesis_results_with_first_12mer.csv"
summary_output_file = "/content/ambush_hypothesis_summary_with_first_12mer.csv"

mappedmer_output_file = "/content/mapped_12mer_stopcodon_analysis.csv"
mappedmer_summary_file = "/content/mapped_12mer_stopcodon_summary.csv"

inferential_output_file = "/content/stop_count_ttest_bonferroni_results.csv"


# ============================================================
# CLEAN FUNCTIONS
# ============================================================

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = seq.replace("U", "T")
    seq = re.sub(r"[^ATGC]", "", seq)
    return seq


def normalize_shift(x):
    x = str(x).strip()

    if x in ["1", "1.0", "+1"]:
        return "1"

    if x in ["2", "2.0", "+2"]:
        return "2"

    if x in ["0", "0.0", "frame0", "inframe", "in_frame"]:
        return "0"

    if x in ["transl_table_issue", "trans_table_issue"]:
        return "transl_table_issue"

    return x


def normalize_transl_table(x):
    try:
        return int(float(str(x).strip()))
    except Exception:
        return None


def clean_position_plus_one(pos):
    try:
        pos = str(pos).strip()

        if "|" in pos:
            pos = pos.split("|")[0]

        return int(float(pos)) + 1

    except Exception:
        return None


def get_shifted_codons(seq, shift):
    shifted_seq = seq[shift:]
    codons = []

    for i in range(0, len(shifted_seq) - 2, 3):
        codon = shifted_seq[i:i + 3]

        if len(codon) == 3:
            codons.append(codon)

    return codons


def find_stop_codons(codons, stop_codons):
    stop_positions = []
    stop_codon_list = []

    for idx, codon in enumerate(codons):
        if codon in stop_codons:
            stop_positions.append(idx + 1)
            stop_codon_list.append(codon)

    return stop_codon_list, stop_positions


def count_stop_codons_before_nt_position(seq, frame_shift, stop_codons, nt_position):
    if nt_position is None or pd.isna(nt_position):
        return 0, "", ""

    codons = get_shifted_codons(seq, frame_shift)

    stop_count = 0
    stop_codon_list = []
    stop_position_list = []

    for idx, codon in enumerate(codons):
        codon_start_0based = frame_shift + idx * 3
        codon_start_1based = codon_start_0based + 1

        if codon_start_1based >= nt_position:
            break

        if codon in stop_codons:
            stop_count += 1
            stop_codon_list.append(codon)
            stop_position_list.append(idx + 1)

    return (
        stop_count,
        "|".join(stop_codon_list),
        "|".join(map(str, stop_position_list))
    )


def count_stop_codons_inside_mer_all_frames(seq_mer, stop_codons):
    seq_mer = clean_sequence(seq_mer)

    stop_count = 0
    found_stop_codons = []
    found_internal_frames = []
    found_positions = []

    for frame in [0, 1, 2]:
        codons = get_shifted_codons(seq_mer, frame)

        for idx, codon in enumerate(codons):
            if codon in stop_codons:
                stop_count += 1
                found_stop_codons.append(codon)
                found_internal_frames.append(str(frame))
                found_positions.append(str(idx + 1))

    return (
        stop_count,
        "|".join(found_stop_codons),
        "|".join(found_internal_frames),
        "|".join(found_positions)
    )


def detect_column(df, possible_columns, required_name):
    for col in possible_columns:
        if col in df.columns:
            print(f"Using {required_name} column: {col}")
            return col

    print("\nAvailable columns:")
    print(df.columns.tolist())

    raise ValueError(
        f"No valid column found for {required_name}. "
        f"Tried: {possible_columns}"
    )


# ============================================================
# LOAD DATA
# ============================================================

print("\nLoading dataset...")

df = pd.read_csv(
    input_file,
    dtype=str,
    low_memory=False
)

print("\nDataset loaded.")
print("Total rows:", len(df))

print("\nColumns found:")
print(df.columns.tolist())


# ============================================================
# AUTO-DETECT REQUIRED COLUMNS
# ============================================================

cds_col = detect_column(
    df,
    [
        "cds_sequence",
        "cds_sequence_clean",
        "CDS_sequence",
        "sequence",
        "Sequence"
    ],
    "CDS sequence"
)

transl_table_col = detect_column(
    df,
    [
        "transl_table",
        "translation_table",
        "transl_table_clean",
        "Translation_table"
    ],
    "translation table"
)

matched_col = detect_column(
    df,
    [
        "matched_12mer",
        "matched_12_mer",
        "matched_15mer",
        "matched_15_mer",
        "matched_sequence",
        "matched_seq",
        "mer12",
        "mer15",
        "sequence_12mer",
        "sequence_15mer",
        "mapped_12mer",
        "mapped_15mer",
        "mapped_sequence"
    ],
    "matched mer"
)
shift_col = None
if "shift" in df.columns:
    shift_col = "shift"
elif "frame_status" in df.columns:
    shift_col = "frame_status"
else:
    print("\nNo shift column found. Creating default shift = unknown.")
    df["shift"] = "unknown"
    shift_col = "shift"

positions_col = None
for col in ["positions", "position", "start", "start_position", "nt_position"]:
    if col in df.columns:
        positions_col = col
        break

if positions_col is None:
    print("\nNo position column found. first mapped 12mer position will be NaN.")
    df["positions"] = np.nan
    positions_col = "positions"


# ============================================================
# CLEAN DATA
# ============================================================

print("\nCleaning dataset...")

df["shift_clean"] = df[shift_col].apply(normalize_shift)
df["cds_sequence_clean"] = df[cds_col].apply(clean_sequence)
df["matched_12mer_clean"] = df[matched_col].apply(clean_sequence)
df["transl_table_clean"] = df[transl_table_col].apply(normalize_transl_table)
df["position_corrected_plus_one"] = df[positions_col].apply(clean_position_plus_one)

df = df[df["cds_sequence_clean"].str.len() > 0].copy()
df = df[df["matched_12mer_clean"].str.len() > 0].copy()

print("\nShift distribution:")
print(df["shift_clean"].value_counts(dropna=False))

print("\nTranslation table distribution:")
print(df["transl_table_clean"].value_counts(dropna=False))

print("\nMatched 12mer length distribution:")
print(df["matched_12mer_clean"].str.len().value_counts().sort_index())


# ============================================================
# BUILD UNIQUE CDS TABLE
# ============================================================

print("\nBuilding unique CDS table...")

unique_cds = df[
    ["cds_sequence_clean", "transl_table_clean"]
].drop_duplicates()

unique_cds = unique_cds.dropna(
    subset=["transl_table_clean"]
)

unique_cds["transl_table_clean"] = unique_cds[
    "transl_table_clean"
].astype(int)

print(
    "Unique CDS + translation table combinations:",
    len(unique_cds)
)


# ============================================================
# COUNT MAPPED MERS AND FIRST MAPPED MER POSITION
# ============================================================

print("\nCounting mapped 12mers and first mapped 12mer position...")

shift1_counts = defaultdict(int)
shift2_counts = defaultdict(int)
frame0_counts = defaultdict(int)
transl_table_issue_counts = defaultdict(int)
all_mapped_counts = defaultdict(int)

first_mapped_position = {}

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Counting mapped 12mers"
):

    cds = row["cds_sequence_clean"]
    shift = row["shift_clean"]
    pos = row["position_corrected_plus_one"]

    if shift == "0":
        frame0_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "1":
        shift1_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "2":
        shift2_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "transl_table_issue":
        transl_table_issue_counts[cds] += 1
        all_mapped_counts[cds] += 1

    else:
        all_mapped_counts[cds] += 1

    if pos is not None and not pd.isna(pos):
        if cds not in first_mapped_position:
            first_mapped_position[cds] = pos
        else:
            first_mapped_position[cds] = min(
                first_mapped_position[cds],
                pos
            )


# ============================================================
# MAIN CDS-LEVEL AMBUSH ANALYSIS
# ============================================================

print(
    "\nRunning original-frame, +1-frame, +2-frame, "
    "and before-first-12mer stop codon analysis..."
)

results = []
invalid_translation_tables = []

for _, row in tqdm(
    unique_cds.iterrows(),
    total=len(unique_cds),
    desc="CDS-level stop codon analysis"
):

    cds_seq = row["cds_sequence_clean"]
    transl_table_id = row["transl_table_clean"]

    try:
        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        invalid_translation_tables.append(transl_table_id)
        continue

    original_codons = get_shifted_codons(cds_seq, 0)
    original_stop_codons, original_stop_positions = find_stop_codons(
        original_codons,
        stop_codons
    )

    plus1_codons = get_shifted_codons(cds_seq, 1)
    plus1_stop_codons, plus1_stop_positions = find_stop_codons(
        plus1_codons,
        stop_codons
    )

    plus2_codons = get_shifted_codons(cds_seq, 2)
    plus2_stop_codons, plus2_stop_positions = find_stop_codons(
        plus2_codons,
        stop_codons
    )

    first_pos = first_mapped_position.get(cds_seq, np.nan)

    original_before_count, original_before_codons, original_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            0,
            stop_codons,
            first_pos
        )
    )

    plus1_before_count, plus1_before_codons, plus1_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            1,
            stop_codons,
            first_pos
        )
    )

    plus2_before_count, plus2_before_codons, plus2_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            2,
            stop_codons,
            first_pos
        )
    )

    results.append({

        "cds_sequence": cds_seq,
        "translation_table": transl_table_id,
        "stop_codons_for_table": "|".join(sorted(stop_codons)),
        "cds_length_nt": len(cds_seq),
        "first_mapped_12mer_nt_position": first_pos,

        "original_total_codons": len(original_codons),
        "original_stop_count": len(original_stop_codons),
        "original_stop_density_per_100_codons": (
            len(original_stop_codons) / len(original_codons) * 100
        ) if len(original_codons) > 0 else np.nan,
        "original_stop_codons": "|".join(original_stop_codons),
        "original_stop_positions": "|".join(map(str, original_stop_positions)),

        "plus1_total_codons": len(plus1_codons),
        "plus1_stop_count": len(plus1_stop_codons),
        "plus1_stop_density_per_100_codons": (
            len(plus1_stop_codons) / len(plus1_codons) * 100
        ) if len(plus1_codons) > 0 else np.nan,
        "plus1_stop_codons": "|".join(plus1_stop_codons),
        "plus1_stop_positions": "|".join(map(str, plus1_stop_positions)),
        "plus1_mapped_12mers": shift1_counts.get(cds_seq, 0),

        "plus2_total_codons": len(plus2_codons),
        "plus2_stop_count": len(plus2_stop_codons),
        "plus2_stop_density_per_100_codons": (
            len(plus2_stop_codons) / len(plus2_codons) * 100
        ) if len(plus2_codons) > 0 else np.nan,
        "plus2_stop_codons": "|".join(plus2_stop_codons),
        "plus2_stop_positions": "|".join(map(str, plus2_stop_positions)),
        "plus2_mapped_12mers": shift2_counts.get(cds_seq, 0),

        "frame0_mapped_12mers": frame0_counts.get(cds_seq, 0),
        "transl_table_issue_mapped_12mers": transl_table_issue_counts.get(cds_seq, 0),

        "total_mapped_12mers_shift1_shift2": (
            shift1_counts.get(cds_seq, 0)
            +
            shift2_counts.get(cds_seq, 0)
        ),

        "total_mapped_12mers_all_tracked_groups": all_mapped_counts.get(cds_seq, 0),

        "original_stop_count_before_first_mapped_12mer": original_before_count,
        "original_stop_codons_before_first_mapped_12mer": original_before_codons,
        "original_stop_positions_before_first_mapped_12mer": original_before_positions,

        "plus1_stop_count_before_first_mapped_12mer": plus1_before_count,
        "plus1_stop_codons_before_first_mapped_12mer": plus1_before_codons,
        "plus1_stop_positions_before_first_mapped_12mer": plus1_before_positions,

        "plus2_stop_count_before_first_mapped_12mer": plus2_before_count,
        "plus2_stop_codons_before_first_mapped_12mer": plus2_before_codons,
        "plus2_stop_positions_before_first_mapped_12mer": plus2_before_positions
    })


# ============================================================
# MAPPED MER-LEVEL STOP CODON ANALYSIS
# ============================================================

print("\nRunning mapped 12mer-level stop codon analysis...")

mappedmer_results = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Mapped 12mer stop codon analysis"
):

    transl_table_id = row["transl_table_clean"]

    if pd.isna(transl_table_id):
        continue

    try:
        transl_table_id = int(transl_table_id)

        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        continue

    seq_mer = row["matched_12mer_clean"]

    stop_count, stop_codons_found, internal_frames, internal_positions = (
        count_stop_codons_inside_mer_all_frames(
            seq_mer,
            stop_codons
        )
    )

    mappedmer_results.append({

        "cds_sequence_clean": row["cds_sequence_clean"],
        "matched_12mer_clean": seq_mer,
        "matched_12mer_length": len(seq_mer),
        "shift": row["shift_clean"],
        "translation_table": transl_table_id,

        "mapped_12mer_has_stop_codon": stop_count > 0,
        "mapped_12mer_stop_codon_count": stop_count,
        "mapped_12mer_stop_codons": stop_codons_found,
        "mapped_12mer_stop_internal_frames": internal_frames,
        "mapped_12mer_stop_internal_positions": internal_positions
    })


# ============================================================
# SAVE DETAILED OUTPUTS
# ============================================================

results_df = pd.DataFrame(results)
mappedmer_df = pd.DataFrame(mappedmer_results)


# ============================================================
# INFERENTIAL STATISTICS
# ============================================================

stop_count_cols = [
    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count"
]

stats_df = results_df[stop_count_cols].copy()

for col in stop_count_cols:
    stats_df[col] = pd.to_numeric(stats_df[col], errors="coerce")

stats_df = stats_df.dropna()

ttest_results = []

for col1, col2 in combinations(stop_count_cols, 2):

    t_stat, p_value = ttest_rel(
        stats_df[col1],
        stats_df[col2],
        nan_policy="omit"
    )

    ttest_results.append({
        "Comparison": f"{col1} vs {col2}",
        "Variable_1": col1,
        "Variable_2": col2,
        "Mean_1": stats_df[col1].mean(),
        "SD_1": stats_df[col1].std(ddof=1),
        "Mean_2": stats_df[col2].mean(),
        "SD_2": stats_df[col2].std(ddof=1),
        "N": len(stats_df),
        "t_statistic": t_stat,
        "raw_p_value": p_value
    })

inferential_df = pd.DataFrame(ttest_results)

if not inferential_df.empty:
    reject, corrected_pvals, _, _ = multipletests(
        inferential_df["raw_p_value"],
        alpha=0.05,
        method="bonferroni"
    )

    inferential_df["bonferroni_corrected_p"] = corrected_pvals
    inferential_df["significant_after_bonferroni_95CI"] = reject


inferential_df.to_csv(
    inferential_output_file,
    index=False
)


# ============================================================
# SAVE MAIN OUTPUT FILES
# ============================================================

results_df.to_csv(
    output_file,
    index=False
)

mappedmer_df.to_csv(
    mappedmer_output_file,
    index=False
)


# ============================================================
# CDS-LEVEL SUMMARY STATISTICS
# ============================================================

summary_cols = [

    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count",

    "original_stop_density_per_100_codons",
    "plus1_stop_density_per_100_codons",
    "plus2_stop_density_per_100_codons",

    "original_stop_count_before_first_mapped_12mer",
    "plus1_stop_count_before_first_mapped_12mer",
    "plus2_stop_count_before_first_mapped_12mer"
]

summary_df = results_df[
    summary_cols
].describe()

summary_df.to_csv(
    summary_output_file
)


# ============================================================
# MAPPED MER STOP SUMMARY
# ============================================================

total_mers = len(mappedmer_df)

stop_containing_mers = mappedmer_df[
    mappedmer_df["mapped_12mer_has_stop_codon"] == True
]

percent_mers_with_stop = (
    len(stop_containing_mers)
    / total_mers
    * 100
    if total_mers > 0
    else np.nan
)

average_stop_codons_all_mers = mappedmer_df[
    "mapped_12mer_stop_codon_count"
].mean()

average_stop_codons_stop_containing_only = stop_containing_mers[
    "mapped_12mer_stop_codon_count"
].mean()

mappedmer_summary = pd.DataFrame({

    "metric": [

        "total_mapped_12mers",
        "mapped_12mers_with_at_least_one_stop",
        "percent_mapped_12mers_with_stop",
        "average_stop_codons_per_12mer_all",
        "average_stop_codons_per_stop_containing_12mer_only"
    ],

    "value": [

        total_mers,
        len(stop_containing_mers),
        percent_mers_with_stop,
        average_stop_codons_all_mers,
        average_stop_codons_stop_containing_only
    ]
})

mappedmer_summary.to_csv(
    mappedmer_summary_file,
    index=False
)


# ============================================================
# FINAL PRINTS
# ============================================================

print("\nInferential statistics saved:")
print(inferential_output_file)

print("\nStop-count paired t-test + Bonferroni results:")
print(inferential_df)

print("\nCDS-level detailed output saved:")
print(output_file)

print("\nMapped 12mer-level output saved:")
print(mappedmer_output_file)

print("\nCDS-level summary saved:")
print(summary_output_file)

print("\nMapped 12mer stop summary saved:")
print(mappedmer_summary_file)

print("\nTotal CDS analyzed:")
print(len(results_df))

print("\nTotal mapped 12mers analyzed:")
print(len(mappedmer_df))

if len(invalid_translation_tables) > 0:
    print("\nInvalid / unsupported translation table IDs encountered:")
    print(sorted(set(invalid_translation_tables)))

print("\nPreview of CDS-level detailed results:")
print(results_df.head())

print("\nCDS-level summary statistics:")
print(summary_df)

print("\nMapped 12mer stop summary:")
print(mappedmer_summary)

print("\nInferential statistics:")
print(inferential_df)


Loading dataset...

Dataset loaded.
Total rows: 440

Columns found:
['uniprot_id', 'nucleotide_id', 'protein_id', 'molecule_type', 'status', 'uniprot_length', 'ncbi_length', 'transl_table', 'codon_start', 'peptide', 'matched_15mer', 'positions', 'position', 'frame_status', 'shift', 'num_hits', 'cds_sequence', 'codon_offset']
Using CDS sequence column: cds_sequence
Using translation table column: transl_table
Using matched mer column: matched_15mer

Cleaning dataset...

Shift distribution:
shift_clean
2    272
1    167
0      1
Name: count, dtype: int64

Translation table distribution:
transl_table_clean
1     295
11    133
4       9
12      2
2       1
Name: count, dtype: int64

Matched 12mer length distribution:
matched_12mer_clean
12    439
24      1
Name: count, dtype: int64

Building unique CDS table...
Unique CDS + translation table combinations: 434

Counting mapped 12mers and first mapped 12mer position...


Counting mapped 12mers: 100%|██████████| 440/440 [00:00<00:00, 11247.52it/s]



Running original-frame, +1-frame, +2-frame, and before-first-12mer stop codon analysis...


CDS-level stop codon analysis: 100%|██████████| 434/434 [00:00<00:00, 457.40it/s]



Running mapped 12mer-level stop codon analysis...


Mapped 12mer stop codon analysis: 100%|██████████| 440/440 [00:00<00:00, 8139.22it/s]



Inferential statistics saved:
/content/stop_count_ttest_bonferroni_results.csv

Stop-count paired t-test + Bonferroni results:
                                Comparison           Variable_1  \
0  original_stop_count vs plus1_stop_count  original_stop_count   
1  original_stop_count vs plus2_stop_count  original_stop_count   
2     plus1_stop_count vs plus2_stop_count     plus1_stop_count   

         Variable_2     Mean_1       SD_1     Mean_2        SD_2    N  \
0  plus1_stop_count   3.274194  34.214524  39.942396   60.945316  434   
1  plus2_stop_count   3.274194  34.214524  69.285714  137.482550  434   
2  plus2_stop_count  39.942396  60.945316  69.285714  137.482550  434   

   t_statistic   raw_p_value  bonferroni_corrected_p  \
0   -13.408875  1.558752e-34            4.676256e-34   
1    -9.803543  1.263195e-20            3.789584e-20   
2    -5.652765  2.868749e-08            8.606247e-08   

   significant_after_bonferroni_95CI  
0                               True  
1      

In [ ]:
### ncORF analysis

In [ ]:
import pandas as pd
from google.colab import files

# Upload the supplementary Excel file
uploaded = files.upload()

# Get uploaded filename
input_file = list(uploaded.keys())[0]
print("Uploaded file:", input_file)

# Read the two main sheets
s2 = pd.read_excel(input_file, sheet_name="S2. PHASE I Ribo-seq ORFs")
s3 = pd.read_excel(input_file, sheet_name="S3. Single-study Ribo-seq ORFs")

# Add source labels
s2["source_sheet"] = "S2_replicated_ORFs"
s3["source_sheet"] = "S3_single_study_ORFs"

# Combine both sheets
combined = pd.concat([s2, s3], ignore_index=True)

# Basic checks
print("S2 rows:", len(s2))
print("S3 rows:", len(s3))
print("Combined rows:", len(combined))
print("Unique ORFs:", combined["orf_name"].nunique())

# Save outputs
combined.to_excel("combined_7264_Riboseq_ORFs.xlsx", index=False)
combined.to_csv("combined_7264_Riboseq_ORFs.csv", index=False)

# Download files
files.download("combined_7264_Riboseq_ORFs.xlsx")
files.download("combined_7264_Riboseq_ORFs.csv")

In [ ]:
import pandas as pd
from google.colab import files
from tqdm.auto import tqdm

# Upload:
# 1. combined_7264_Riboseq_ORFs.csv
# 2. final_absent_pep_tranls.csv
uploaded = files.upload()

orf_file = "combined_7264_Riboseq_ORFs.csv"
pep_file = "final_absent_pep_tranls.csv"

print("Reading files...")
orf_df = pd.read_csv(orf_file)
pep_df = pd.read_csv(pep_file)

print("ORF dataframe:", orf_df.shape)
print("Peptide dataframe:", pep_df.shape)
print("Peptide columns:", pep_df.columns.tolist())

peptide_col = "pentapeptide"

# Clean peptide list
pep_df[peptide_col] = (
    pep_df[peptide_col]
    .astype(str)
    .str.strip()
    .str.upper()
)

query_peptides = sorted(set(pep_df[peptide_col].dropna()))

query_peptides = [
    p for p in query_peptides
    if len(p) == 5 and p.isalpha()
]

query_set = set(query_peptides)

print("Unique valid pentapeptides:", len(query_set))

# Clean ORF sequences
orf_df["orf_sequence"] = (
    orf_df["orf_sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

matches = []

print("\nSearching pentapeptides in ncORF sequences...\n")

for idx, row in tqdm(
    orf_df.iterrows(),
    total=len(orf_df),
    desc="Scanning 7,264 ncORFs"
):
    seq = row["orf_sequence"]

    if not isinstance(seq, str) or seq == "" or seq == "NAN":
        continue

    seq_len = len(seq)

    for i in range(seq_len - 4):
        pentamer = seq[i:i+5]

        if pentamer in query_set:
            context_start = max(0, i - 10)
            context_end = min(seq_len, i + 5 + 10)

            matches.append({
                "matched_peptide": pentamer,
                "orf_name": row.get("orf_name", ""),
                "orf_biotype": row.get("orf_biotype", ""),
                "gene_name": row.get("gene_name", ""),
                "gene_id": row.get("gene_id", ""),
                "transcript": row.get("transcript", ""),
                "chrm": row.get("chrm", ""),
                "strand": row.get("strand", ""),
                "orf_length": row.get("orf_length", seq_len),
                "match_start_1based": i + 1,
                "match_end_1based": i + 5,
                "context_sequence": seq[context_start:context_end],
                "MS_peptides": row.get("MS_peptides", ""),
                "number_studies_MS": row.get("number_studies_MS", ""),
                "Riboseq_evidence": row.get("Riboseq_evidence", ""),
                "source_sheet": row.get("source_sheet", "")
            })

matches_df = pd.DataFrame(matches)

matched_peptides = set(matches_df["matched_peptide"]) if len(matches_df) > 0 else set()
unmatched_peptides = sorted(query_set - matched_peptides)

summary_df = pd.DataFrame([{
    "total_query_pentapeptides": len(query_set),
    "matched_unique_pentapeptides": len(matched_peptides),
    "unmatched_unique_pentapeptides": len(unmatched_peptides),
    "total_match_events": len(matches_df),
    "unique_ncORFs_with_matches": matches_df["orf_name"].nunique() if len(matches_df) > 0 else 0,
    "unique_genes_with_matches": matches_df["gene_name"].nunique() if len(matches_df) > 0 else 0
}])

unmatched_df = pd.DataFrame({
    "unmatched_peptide": unmatched_peptides
})

print("\nSUMMARY")
display(summary_df)

print("\nTop matches")
display(matches_df.head())

# Save outputs
matches_df.to_csv("final_absent_pentapeptides_matched_in_7264_ncORFs.csv", index=False)
unmatched_df.to_csv("final_absent_pentapeptides_not_found_in_7264_ncORFs.csv", index=False)
summary_df.to_csv("final_absent_pentapeptide_ncORF_search_summary.csv", index=False)

files.download("final_absent_pentapeptides_matched_in_7264_ncORFs.csv")
files.download("final_absent_pentapeptides_not_found_in_7264_ncORFs.csv")
files.download("final_absent_pentapeptide_ncORF_search_summary.csv")

print("DONE.")

In [ ]:
# ============================================================
# ADD 2026 PROTEIN-LEVEL EVIDENCE ANNOTATION
# ============================================================

# Upload the 2026 supplementary files:
# 1. 41586_2026_10459_MOESM13_ESM.xlsx  -> ORF-level protein evidence
# 2. 41586_2026_10459_MOESM14_ESM.xlsx  -> peptide-level MS evidence

print("\nUpload 2026 supplementary evidence files...")
uploaded_evidence = files.upload()

# File names
orf_evidence_file = "41586_2026_10459_MOESM13_ESM.xlsx"
peptide_evidence_file = "41586_2026_10459_MOESM14_ESM.xlsx"

# Read ORF-level evidence file
evidence_orf_df = pd.read_excel(
    orf_evidence_file,
    sheet_name="Table S3"
)

# Read peptide-level evidence file
evidence_peptide_df = pd.read_excel(
    peptide_evidence_file,
    sheet_name="Table S2"
)

# Clean key columns
matches_df["orf_name"] = matches_df["orf_name"].astype(str).str.strip()
evidence_orf_df["Ribo-Seq_ORF"] = evidence_orf_df["Ribo-Seq_ORF"].astype(str).str.strip()
evidence_peptide_df["Ribo-Seq_ORF"] = evidence_peptide_df["Ribo-Seq_ORF"].astype(str).str.strip()

# ============================================================
# MATCH 703 PEPTIDE-HARBORING ncORFs WITH 2026 PROTEIN EVIDENCE
# ============================================================

protein_level_matches = matches_df.merge(
    evidence_orf_df,
    left_on="orf_name",
    right_on="Ribo-Seq_ORF",
    how="inner",
    suffixes=("_matched", "_2026")
)

print("\nProtein-level evidence matches found:")
print("Total match rows:", len(protein_level_matches))
print("Unique matched peptides:", protein_level_matches["matched_peptide"].nunique())
print("Unique ncORFs with protein-level evidence:", protein_level_matches["orf_name"].nunique())

# ============================================================
# DEFINE HIGH-CONFIDENCE EVIDENCE
# Here, Tier 2A is treated as high-confidence non-HLA protein-level evidence
# ============================================================

protein_level_matches["Final.tier.for.level.of.evidence"] = (
    protein_level_matches["Final.tier.for.level.of.evidence"]
    .astype(str)
    .str.strip()
)

high_confidence_matches = protein_level_matches[
    protein_level_matches["Final.tier.for.level.of.evidence"].str.contains(
        "Tier 2A",
        case=False,
        na=False
    )
].copy()

print("\nHigh-confidence protein-level evidence matches:")
print("Total high-confidence rows:", len(high_confidence_matches))
print("Unique high-confidence peptides:", high_confidence_matches["matched_peptide"].nunique())
print("Unique high-confidence ncORFs:", high_confidence_matches["orf_name"].nunique())

# ============================================================
# ADD PEPTIDE-LEVEL MS DETAILS FROM TABLE S2
# ============================================================

protein_level_with_ms_details = protein_level_matches.merge(
    evidence_peptide_df,
    left_on="orf_name",
    right_on="Ribo-Seq_ORF",
    how="left",
    suffixes=("", "_peptideMS")
)

high_confidence_with_ms_details = high_confidence_matches.merge(
    evidence_peptide_df,
    left_on="orf_name",
    right_on="Ribo-Seq_ORF",
    how="left",
    suffixes=("", "_peptideMS")
)

# ============================================================
# SAVE OUTPUT FILES
# ============================================================

protein_level_matches.to_csv(
    "703_peptides_with_2026_protein_level_ORF_evidence.csv",
    index=False
)

high_confidence_matches.to_csv(
    "703_peptides_with_2026_high_confidence_Tier2A_evidence.csv",
    index=False
)

protein_level_with_ms_details.to_csv(
    "703_peptides_with_2026_protein_level_and_MS_peptide_details.csv",
    index=False
)

high_confidence_with_ms_details.to_csv(
    "703_peptides_with_2026_high_confidence_and_MS_peptide_details.csv",
    index=False
)

# ============================================================
# SUMMARY TABLE
# ============================================================

evidence_summary_df = pd.DataFrame([{
    "total_ncORF_matched_peptides_from_703_set": matches_df["matched_peptide"].nunique(),
    "total_ncORFs_with_703_peptide_matches": matches_df["orf_name"].nunique(),

    "unique_peptides_with_2026_protein_level_evidence":
        protein_level_matches["matched_peptide"].nunique(),

    "unique_ncORFs_with_2026_protein_level_evidence":
        protein_level_matches["orf_name"].nunique(),

    "unique_peptides_with_high_confidence_Tier2A_evidence":
        high_confidence_matches["matched_peptide"].nunique(),

    "unique_ncORFs_with_high_confidence_Tier2A_evidence":
        high_confidence_matches["orf_name"].nunique()
}])

display(evidence_summary_df)

evidence_summary_df.to_csv(
    "703_peptide_2026_protein_evidence_summary.csv",
    index=False
)

# Download outputs
files.download("703_peptides_with_2026_protein_level_ORF_evidence.csv")
files.download("703_peptides_with_2026_high_confidence_Tier2A_evidence.csv")
files.download("703_peptides_with_2026_protein_level_and_MS_peptide_details.csv")
files.download("703_peptides_with_2026_high_confidence_and_MS_peptide_details.csv")
files.download("703_peptide_2026_protein_evidence_summary.csv")

print("\n2026 protein-level evidence extraction complete.")